# Lab 1 — IT Operations Maturity Assessment & AIOps Use-Case Identification
### AIOps Professional 2026 | Oracle Cloud Infrastructure

---

## Objective
By the end of this lab you will:
- Visualise **real-looking OCI operational data** — metrics, alerts, and incidents
- Identify **patterns that humans miss** but AIOps can detect
- Score your IT Operations against the **AIOps Maturity Model**
- Map your gaps to specific **AIOps use cases**

> **No prior Python experience needed.** Run each cell top to bottom with `Shift + Enter`.

---

## Section 1 — Setup
Run this cell first. It loads all libraries and sets up the lab environment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Reproducible randomness
np.random.seed(42)

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#F8F9FA',
    'axes.facecolor':   '#FFFFFF',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.family':      'sans-serif',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

print('✅  Environment ready.')

---
## Section 2 — Simulated OCI Environment

The cell below generates **72 hours of realistic operational data** for a typical OCI environment:
- 4 compute instances (`web-01` to `web-04`)
- 1 OKE cluster with 3 pods
- 1 Oracle DB (Autonomous)
- A Load Balancer

This is the kind of raw telemetry that lands in **OCI Monitoring** every minute.

**Run the cell — the data is generated, not fetched from any server.**

In [ ]:
# ── Time axis: 72 hours at 1-minute resolution ────────────────────────────────
start = datetime(2026, 4, 10, 0, 0, 0)
minutes = 72 * 60  # 4320 points
timestamps = [start + timedelta(minutes=i) for i in range(minutes)]
t = np.arange(minutes)

# ── Helper: add realistic daily pattern ───────────────────────────────────────
def daily_pattern(t, peak_hour=14, amplitude=1.0):
    """Business-hours load curve — peaks at peak_hour."""
    hour = (t / 60) % 24
    return amplitude * np.exp(-0.5 * ((hour - peak_hour) / 4) ** 2)

# ── CPU metrics (4 compute instances) ────────────────────────────────────────
instances = ['web-01', 'web-02', 'web-03', 'web-04']
cpu = {}
for idx, inst in enumerate(instances):
    base = 25 + idx * 5
    trend = daily_pattern(t, peak_hour=14, amplitude=30)
    noise = np.random.normal(0, 3, minutes)
    series = base + trend + noise
    # Inject a real incident: web-02 spikes at hour 38 for 45 minutes
    if inst == 'web-02':
        spike_start = 38 * 60
        series[spike_start:spike_start + 45] += np.linspace(0, 55, 45)
        series[spike_start + 45:spike_start + 90] += np.linspace(55, 0, 45)
    # web-04 has gradual memory leak pattern starting hour 48
    if inst == 'web-04':
        leak_start = 48 * 60
        series[leak_start:] += np.linspace(0, 40, minutes - leak_start)
    cpu[inst] = np.clip(series, 0, 100)

# ── Alert log (simulated OCI Alarms output) ───────────────────────────────────
alert_data = [
    # (hour, resource,              severity,  message)
    (6,   'web-02',                 'WARNING',  'CPU > 70%'),
    (6,   'Load Balancer',          'WARNING',  'Latency > 200ms'),
    (6,   'web-01',                 'INFO',     'Memory usage 65%'),
    (6,   'OKE: pod-api-7f9b',      'WARNING',  'CPU throttling detected'),
    (6,   'OKE: pod-api-3c2a',      'INFO',     'Pod restart count: 2'),
    (6,   'Autonomous DB',          'INFO',     'Connection pool 78% utilised'),
    (38,  'web-02',                 'CRITICAL', 'CPU > 95%'),
    (38,  'web-02',                 'CRITICAL', 'Response time > 5s'),
    (38,  'Load Balancer',          'CRITICAL', 'Backend web-02 unhealthy'),
    (38,  'OKE: pod-api-7f9b',      'CRITICAL', 'OOMKilled'),
    (38,  'OKE: pod-worker-1a3f',   'WARNING',  'High memory usage'),
    (38,  'Autonomous DB',          'CRITICAL', 'Connection wait time > 3s'),
    (38,  'Autonomous DB',          'WARNING',  'Long-running query detected'),
    (38,  'web-01',                 'WARNING',  'CPU > 75% (load shifted from web-02)'),
    (38,  'web-03',                 'WARNING',  'CPU > 72% (load shifted from web-02)'),
    (48,  'web-04',                 'WARNING',  'Memory usage trending up'),
    (52,  'web-04',                 'CRITICAL', 'Memory > 85%'),
    (56,  'web-04',                 'CRITICAL', 'OOM — process killed'),
    (60,  'OKE: pod-api-7f9b',      'INFO',     'Pod restarted successfully'),
]

alerts_df = pd.DataFrame(alert_data, columns=['hour', 'resource', 'severity', 'message'])
alerts_df['time'] = alerts_df['hour'].apply(lambda h: start + timedelta(hours=h))

# ── Incident log (what an NOC operator manually recorded) ─────────────────────
incidents = pd.DataFrame([
    {'id':'INC-1001', 'opened': start+timedelta(hours=38,minutes=12), 'closed': start+timedelta(hours=39,minutes=45),
     'resource':'web-02', 'action':'Operator restarted app server manually', 'MTTR_min': 93},
    {'id':'INC-1002', 'opened': start+timedelta(hours=56,minutes=3),  'closed': start+timedelta(hours=58,minutes=30),
     'resource':'web-04', 'action':'Ops team cleared memory — root cause unknown', 'MTTR_min': 147},
])

print(f'✅  Generated {minutes:,} data points across {len(instances)} instances.')
print(f'    Alerts: {len(alerts_df)}   |   Incidents: {len(incidents)}')
print(f'    Time window: {start.strftime("%d %b %Y %H:%M")}  →  {timestamps[-1].strftime("%d %b %Y %H:%M")}')

---
## Section 3 — Visualise the Raw Telemetry

This is what an **OCI Monitoring dashboard** would show your NOC team.

**Before you run the cell, answer this question:**
> *"If you were an on-call engineer watching this data in real time, what would you notice — and what would you miss?"*

Run the cell, then read the observations below the chart.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex=True)
fig.suptitle('OCI Compute CPU % — 72-Hour Window (web-01 to web-04)', fontsize=14, fontweight='bold', y=1.01)

colours = ['#2E75B6', '#C00000', '#375623', '#7030A0']
threshold_warn = 70
threshold_crit = 90

for ax, (inst, colour) in zip(axes.flat, zip(instances, colours)):
    ax.plot(t / 60, cpu[inst], color=colour, linewidth=0.8, alpha=0.9)
    ax.axhline(threshold_warn, color='orange', linewidth=1, linestyle='--', label='Warn 70%')
    ax.axhline(threshold_crit, color='red',    linewidth=1, linestyle='--', label='Crit 90%')
    ax.set_title(inst, fontweight='bold', color=colour)
    ax.set_ylabel('CPU %')
    ax.set_ylim(0, 110)
    ax.set_xlim(0, 72)
    ax.set_xticks(range(0, 73, 6))
    ax.set_xticklabels([f'{h}h' for h in range(0, 73, 6)])
    if inst == 'web-02':
        ax.axvspan(38, 39.5, alpha=0.15, color='red', label='Incident INC-1001')
        ax.annotate('INC-1001', xy=(38.5, 98), fontsize=8, color='red', ha='center')
    if inst == 'web-04':
        ax.axvspan(48, 72, alpha=0.08, color='purple', label='Gradual leak')
        ax.annotate('Memory leak\n(gradual)', xy=(60, 95), fontsize=8, color='purple', ha='center')

axes[1][0].set_xlabel('Hours from start')
axes[1][1].set_xlabel('Hours from start')
plt.tight_layout()
plt.savefig('section3_raw_telemetry.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: section3_raw_telemetry.png')

### 🔍 What did you notice?

| What you likely saw | What you likely missed |
|---|---|
| The `web-02` spike at hour 38 — it is obvious in hindsight | `web-01` and `web-03` also spiked at hour 38 — load shifted from `web-02`. This is **the root cause hiding in plain sight** |
| The `web-04` memory leak starting at hour 48 | The leak started **24 hours before the OOM kill**. An alert only fired at hour 56. AIOps would predict it at hour 50 |
| 19 alerts fired during the 72-hour window | 14 of those 19 alerts were **caused by a single root cause** (INC-1001). A human gets 14 pages. AIOps sends 1 |

> **This is the AIOps value proposition in data.**

---
## Section 4 — The Alert Storm Problem

Print the full alert log — exactly what your on-call engineer would see in their inbox during INC-1001.

In [ ]:
# ── Colour-code severity ───────────────────────────────────────────────────────
sev_icon = {'CRITICAL': '🔴', 'WARNING': '🟡', 'INFO': '🔵'}

print('=' * 70)
print('  OCI ALARMS — 72-HOUR ALERT LOG')
print('=' * 70)

for _, row in alerts_df.iterrows():
    icon = sev_icon.get(row['severity'], '⚪')
    print(f"  {icon} [{row['time'].strftime('%d %b %H:%M')}]  {row['severity']:<8}  {row['resource']:<28}  {row['message']}")

print('=' * 70)

# ── How many were noise? ───────────────────────────────────────────────────────
inc1001_alerts = alerts_df[alerts_df['hour'] == 38]
print(f'\n  Total alerts:          {len(alerts_df)}')
print(f'  Alerts at INC-1001:    {len(inc1001_alerts)}  ← all from 1 root cause')
print(f'  Actual incidents:      2')
print(f'  Alert-to-incident ratio: {len(alerts_df)/2:.1f}:1  ← alert fatigue')
print(f'\n  An AIOps platform correlates these to: 2 correlated events.')

### 🔍 Observation

During INC-1001, **8 alerts fired in the same minute** from 5 different resources — all caused by one overloaded `web-02` node.

A traditional NOC team:
- Gets paged 8 times
- Opens 8 tickets
- Investigates each resource separately
- Resolves `web-02` last — after chasing the symptoms for 40 minutes

AIOps noise reduction (Session 8 of this course) collapses these 8 alerts into **1 correlated event** pointing directly at `web-02`.

---
## Section 5 — MTTR Analysis

Calculate how long it took to resolve each incident — and what AIOps could have done differently.

In [ ]:
print('=' * 70)
print('  INCIDENT SUMMARY')
print('=' * 70)
for _, inc in incidents.iterrows():
    print(f"\n  Incident   : {inc['id']}")
    print(f"  Resource   : {inc['resource']}")
    print(f"  Opened     : {inc['opened'].strftime('%d %b %H:%M')}")
    print(f"  Closed     : {inc['closed'].strftime('%d %b %H:%M')}")
    print(f"  MTTR       : {inc['MTTR_min']} minutes")
    print(f"  Resolution : {inc['action']}")
print('=' * 70)

# ── Plot: Traditional vs AIOps MTTR ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

incidents_plot = incidents.copy()
# AIOps predicted the web-04 leak earlier — MTTR cut significantly
aiops_mttr = [35, 30]  # AIOps automated detection + remediation
traditional_mttr = [93, 147]

x = np.arange(2)
width = 0.35
b1 = ax.bar(x - width/2, traditional_mttr, width, label='Traditional NOC',  color='#C00000', alpha=0.85)
b2 = ax.bar(x + width/2, aiops_mttr,       width, label='With AIOps',       color='#375623', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f"{inc['id']}\n({inc['resource']})" for _, inc in incidents.iterrows()], fontsize=11)
ax.set_ylabel('MTTR (minutes)')
ax.set_title('Mean Time to Resolve — Traditional NOC vs AIOps', fontsize=13, fontweight='bold')
ax.legend()

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{int(bar.get_height())}m', ha='center', fontsize=10, color='#C00000', fontweight='bold')
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{int(bar.get_height())}m', ha='center', fontsize=10, color='#375623', fontweight='bold')

reduction_1 = round((1 - aiops_mttr[0]/traditional_mttr[0]) * 100)
reduction_2 = round((1 - aiops_mttr[1]/traditional_mttr[1]) * 100)
ax.annotate(f'↓ {reduction_1}% reduction', xy=(0+width/2, aiops_mttr[0]+10), xytext=(0.4, aiops_mttr[0]+35),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)
ax.annotate(f'↓ {reduction_2}% reduction', xy=(1+width/2, aiops_mttr[1]+10), xytext=(1.4, aiops_mttr[1]+40),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)

plt.tight_layout()
plt.savefig('section5_mttr.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: section5_mttr.png')

---
## Section 6 — AIOps Maturity Assessment

Rate your current IT Operations organisation against the 5 AIOps maturity dimensions.

### Instructions
In the cell below, change the score for **each dimension** to reflect your organisation (1 = lowest, 5 = highest).
Then run the cell to see your maturity radar chart.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  CHANGE THESE SCORES (1 = lowest / 5 = highest)
# ─────────────────────────────────────────────────────────────────────────────

my_scores = {
    'Data Collection\n& Telemetry':      3,   # How well do you collect logs/metrics/traces?
    'Alerting\n& Noise Reduction':       2,   # How many false/duplicate alerts do you get?
    'Root Cause\nAnalysis':              2,   # Is RCA manual or assisted?
    'Automation\n& Remediation':         1,   # How much is auto-remediated vs manual?
    'Prediction\n& Capacity Planning':   1,   # Do you predict failures or react to them?
}

# ─────────────────────────────────────────────────────────────────────────────

print('Your AIOps Maturity Scores:')
print('-' * 45)
for dim, score in my_scores.items():
    bar = '█' * score + '░' * (5 - score)
    label = dim.replace('\n', ' ')
    print(f'  {label:<38}  {bar}  {score}/5')
print('-' * 45)
avg = sum(my_scores.values()) / len(my_scores)
print(f'  Average score: {avg:.1f}/5')

# Maturity level
if avg < 1.5:
    level = 'Level 1 — Reactive'
elif avg < 2.5:
    level = 'Level 2 — Aware'
elif avg < 3.5:
    level = 'Level 3 — Proactive'
elif avg < 4.5:
    level = 'Level 4 — Predictive'
else:
    level = 'Level 5 — Autonomous'
print(f'  Maturity Level: {level}')

In [ ]:
# ── Radar chart ───────────────────────────────────────────────────────────────
labels = list(my_scores.keys())
values = list(my_scores.values())
n = len(labels)

# target (industry average for organisations adopting AIOps)
target = [4, 4, 3, 3, 3]

angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
values_plot  = values  + [values[0]]
target_plot  = target  + [target[0]]
angles_plot  = angles  + [angles[0]]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#F8F9FA')

# Target area
ax.fill(angles_plot, target_plot, alpha=0.1, color='#2E75B6')
ax.plot(angles_plot, target_plot, color='#2E75B6', linewidth=1.5, linestyle='--', label='AIOps Adoption Target')

# My scores
ax.fill(angles_plot, values_plot, alpha=0.25, color='#C00000')
ax.plot(angles_plot, values_plot, color='#C00000', linewidth=2.5, marker='o', markersize=8, label='Your Organisation')

ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1\nReactive', '2\nAware', '3\nProactive', '4\nPredictive', '5\nAutonomous'], fontsize=8, color='gray')
ax.set_xticks(angles)
ax.set_xticklabels(labels, fontsize=11, fontweight='bold')
ax.set_title('AIOps Maturity Radar', fontsize=15, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=10)

plt.tight_layout()
plt.savefig('section6_maturity_radar.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: section6_maturity_radar.png')

---
## Section 7 — AIOps Use-Case Identification

Based on your maturity scores, the cell below automatically maps your **lowest-scoring gaps** to the AIOps use cases covered in this course.

This is the deliverable you take back to your team.

In [ ]:
# ── Use-case map keyed to maturity dimension ──────────────────────────────────
use_case_map = {
    'Data Collection\n& Telemetry': [
        ('AIOps Session 3',  'IT Operations Data Sources — logs, metrics, traces, events'),
        ('AIOps Session 4',  'OCI Observability Services — Logging Analytics, Monitoring, Alarms'),
        ('AIOps Lab 3',      'Working with Kubernetes — collecting K8s telemetry'),
        ('AIOps Lab 4',      'Create and Manage Logs with OCI Logging Service'),
    ],
    'Alerting\n& Noise Reduction': [
        ('AIOps Session 8',  'Noise Reduction & Alert Correlation'),
        ('AIOps Lab 8',      'Alert Noise Reduction and Event Correlation'),
        ('AIOps Lab 6',      'Define Rules that Trigger Actions on DevOps Events'),
    ],
    'Root Cause\nAnalysis': [
        ('AIOps Session 12', 'Root Cause Analysis — event correlation, causal analysis, topology-aware RCA'),
        ('AIOps Lab 12',     'Automated RCA for Kubernetes Incidents'),
        ('AIOps Session 11', 'Vector Databases — storing and querying past incident embeddings for RCA'),
    ],
    'Automation\n& Remediation': [
        ('AIOps Session 13', 'Automated Remediation & Self-Healing'),
        ('AIOps Session 14', 'CI/CD pipeline integration for closed-loop remediation'),
        ('AIOps Lab 13',     'Automate Web App Deployment using OCI DevOps CI/CD'),
        ('AIOps Lab 14',     'Build Microservices using OCI DevOps'),
        ('AIOps Session 16', 'LangChain — building custom AIOps automation chains'),
    ],
    'Prediction\n& Capacity Planning': [
        ('AIOps Session 10', 'Anomaly Detection — threshold-based vs ML-based'),
        ('AIOps Lab 10',     'Metric-Based Anomaly Detection'),
        ('AIOps Session 9',  'Prompt Engineering — querying LLMs for operational forecasts'),
        ('AIOps Lab 9',      'Getting Started with Prompt Engineering'),
    ],
}

# ── Print prioritised use cases based on your gaps ────────────────────────────
print('=' * 70)
print('  YOUR AIOPS USE-CASE PRIORITIES  (lowest maturity score first)')
print('=' * 70)

sorted_dims = sorted(my_scores.items(), key=lambda x: x[1])

for rank, (dim, score) in enumerate(sorted_dims, 1):
    gap = 5 - score
    label = dim.replace('\n', ' ')
    print(f'\n  Priority {rank}  |  {label}')
    print(f'  Current score: {score}/5  |  Gap to target: {gap} points')
    print('  Relevant course content:')
    for session, desc in use_case_map[dim]:
        print(f'    → {session:<18} {desc}')

print('\n' + '=' * 70)
print('  RECOMMENDED FOCUS AREAS FOR YOUR ORGANISATION')
print('=' * 70)
top3 = [dim.replace('\n', ' ') for dim, _ in sorted_dims[:3]]
for i, dim in enumerate(top3, 1):
    print(f'  {i}. {dim}')

In [ ]:
# ── Gap bar chart ─────────────────────────────────────────────────────────────
dims_clean   = [d.replace('\n', ' ') for d in my_scores.keys()]
scores_vals  = list(my_scores.values())
target_vals  = [4, 4, 3, 3, 3]
gaps         = [t - s for s, t in zip(scores_vals, target_vals)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Your AIOps Gap Analysis', fontsize=14, fontweight='bold')

# Left: current vs target
x = np.arange(len(dims_clean))
w = 0.35
ax1.bar(x - w/2, scores_vals, w, label='Your score',    color='#C00000', alpha=0.85)
ax1.bar(x + w/2, target_vals, w, label='Target (AIOps Adoption)', color='#2E75B6', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(dims_clean, rotation=15, ha='right', fontsize=9)
ax1.set_ylim(0, 5.5)
ax1.set_ylabel('Maturity Score')
ax1.set_title('Current Score vs Target')
ax1.legend()
for i, (s, t) in enumerate(zip(scores_vals, target_vals)):
    ax1.text(i - w/2, s + 0.1, str(s), ha='center', fontsize=9, color='#C00000', fontweight='bold')
    ax1.text(i + w/2, t + 0.1, str(t), ha='center', fontsize=9, color='#2E75B6', fontweight='bold')

# Right: gap = effort needed
gap_colours = ['#C00000' if g >= 3 else '#FF8C00' if g >= 2 else '#375623' for g in gaps]
bars = ax2.barh(dims_clean, gaps, color=gap_colours, alpha=0.85)
ax2.set_xlabel('Gap (points to target)')
ax2.set_title('AIOps Investment Priority (larger gap = higher priority)')
ax2.set_xlim(0, 5)
for bar, gap in zip(bars, gaps):
    if gap > 0:
        ax2.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 f'Gap: {gap}', va='center', fontsize=9, fontweight='bold')

red_p   = mpatches.Patch(color='#C00000', alpha=0.85, label='High priority (gap ≥ 3)')
amber_p = mpatches.Patch(color='#FF8C00', alpha=0.85, label='Medium priority (gap = 2)')
green_p = mpatches.Patch(color='#375623', alpha=0.85, label='Lower priority (gap ≤ 1)')
ax2.legend(handles=[red_p, amber_p, green_p], fontsize=9)

plt.tight_layout()
plt.savefig('section7_gap_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: section7_gap_analysis.png')

---
## Section 8 — Discussion: What Would AIOps Have Done Differently?

Run the summary cell below, then discuss with your group.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║         WHAT AIOPS WOULD HAVE DONE — INC-1001 WALKTHROUGH           ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Hour 37:50  AIOps detects CPU on web-02 rising above baseline       ║
║              (anomaly detection — 10 min before first alert fires)   ║
║                                                                      ║
║  Hour 38:00  8 alerts fire across 5 resources                        ║
║              AIOps correlates → 1 event:  ROOT CAUSE = web-02        ║
║                                                                      ║
║  Hour 38:01  Self-healing: OCI DevOps pipeline triggered             ║
║              → Restart web-02 app process automatically              ║
║              → Shift 20% load to web-05 (spare capacity)             ║
║                                                                      ║
║  Hour 38:05  Incident closed. MTTR: 5 minutes.                       ║
║              (Traditional NOC MTTR: 93 minutes)                      ║
║                                                                      ║
║  Hour 50:00  AIOps predicts web-04 OOM in ~6 hours                   ║
║              (trend analysis on memory leak pattern)                 ║
║              → Ticket raised. Engineer reviews — not paged at 3am.   ║
║                                                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║  This course will teach you to build every one of these steps.       ║
╚══════════════════════════════════════════════════════════════════════╝
""")

print('Discussion questions for your group:')
print('  1. Which of these AIOps capabilities does your team currently have?')
print('  2. Which incident in the last 6 months would have been resolved faster with AIOps?')
print('  3. What is YOUR highest-priority use case from the gap analysis above?')

---
## ✅ Lab Complete

### What you did in this lab

| Step | What you saw |
|---|---|
| Section 2 | Generated 72 hours of realistic OCI compute telemetry |
| Section 3 | Visualised raw CPU metrics — spotted the obvious and the hidden patterns |
| Section 4 | Saw the alert storm problem — 14 alerts from 1 root cause |
| Section 5 | Quantified MTTR reduction with AIOps — 62–80% faster resolution |
| Section 6 | Scored your IT Operations against the AIOps Maturity Model |
| Section 7 | Mapped your gaps to specific sessions and labs in this course |
| Section 8 | Walked through what AIOps would have done for INC-1001 |

### Files generated
- `section3_raw_telemetry.png`
- `section5_mttr.png`
- `section6_maturity_radar.png`
- `section7_gap_analysis.png`

---
**Next: Session 2 — AIOps Reference Architecture**